In [16]:
!pip install -q gradio pandas numpy matplotlib

In [17]:
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import string
import os
from datetime import datetime
from IPython.display import display

print("Cloud-Based Smart Parking System")
print("System initialized successfully!")

Cloud-Based Smart Parking System
System initialized successfully!


In [18]:
NUM_SLOTS = 20

parking_slots = pd.DataFrame({
    "Slot_ID": [f"S{i:02d}" for i in range(1, NUM_SLOTS + 1)],
    "Status": ["Available"] * NUM_SLOTS,
    "Vehicle_Number": ["-"] * NUM_SLOTS,
    "Entry_Time": ["-"] * NUM_SLOTS,
    "Exit_Time": ["-"] * NUM_SLOTS,
    "Parking_Hours": [0.0] * NUM_SLOTS,
    "Parking_Fee": [0.0] * NUM_SLOTS
})

parking_history = pd.DataFrame(columns=[
    "Vehicle_Number",
    "Slot_ID",
    "Entry_Time",
    "Exit_Time",
    "Parking_Hours",
    "Parking_Fee"
])

print("Parking database created.")

Parking database created.


In [19]:
def generate_vehicle_number():
    state = random.choice(["TN", "KA", "KL", "AP"])
    district = random.randint(1, 99)
    letters = ''.join(random.choices(string.ascii_uppercase, k=2))
    number = random.randint(1000, 9999)

    return f"{state}{district:02d}{letters}{number}"


def calculate_fee(hours):
    if hours <= 1:
        return 20

    additional_hours = int(np.ceil(hours - 1))

    return 20 + additional_hours * 10


def get_available_slots():
    return parking_slots[
        parking_slots["Status"] == "Available"
    ]["Slot_ID"].tolist()


def get_occupied_slots():
    return parking_slots[
        parking_slots["Status"] == "Occupied"
    ]["Slot_ID"].tolist()

In [20]:
def get_dashboard():

    total = len(parking_slots)

    available = len(
        parking_slots[
            parking_slots["Status"] == "Available"
        ]
    )

    occupied = total - available

    occupancy_rate = (occupied / total) * 100

    revenue = parking_history["Parking_Fee"].sum()

    return (
        total,
        available,
        occupied,
        occupancy_rate,
        revenue
    )

In [21]:
def vehicle_entry(vehicle_number):

    global parking_slots

    vehicle_number = vehicle_number.strip().upper()

    if vehicle_number == "":
        vehicle_number = generate_vehicle_number()

    existing_vehicle = parking_slots[
        parking_slots["Vehicle_Number"] == vehicle_number
    ]

    if len(existing_vehicle) > 0:
        return (
            f"Vehicle {vehicle_number} is already parked.",
            parking_slots
        )

    available = parking_slots[
        parking_slots["Status"] == "Available"
    ]

    if len(available) == 0:
        return (
            "No parking slots are currently available.",
            parking_slots
        )

    slot_id = available.iloc[0]["Slot_ID"]

    entry_time = datetime.now()

    parking_slots.loc[
        parking_slots["Slot_ID"] == slot_id,
        "Status"
    ] = "Occupied"

    parking_slots.loc[
        parking_slots["Slot_ID"] == slot_id,
        "Vehicle_Number"
    ] = vehicle_number

    parking_slots.loc[
        parking_slots["Slot_ID"] == slot_id,
        "Entry_Time"
    ] = entry_time.strftime("%Y-%m-%d %H:%M:%S")

    parking_slots.loc[
        parking_slots["Slot_ID"] == slot_id,
        "Exit_Time"
    ] = "-"

    message = (
        f"Vehicle Entry Successful!\n\n"
        f"Vehicle Number: {vehicle_number}\n"
        f"Allocated Slot: {slot_id}\n"
        f"Entry Time: {entry_time.strftime('%Y-%m-%d %H:%M:%S')}"
    )

    return message, parking_slots

In [22]:
def vehicle_exit(vehicle_number):

    global parking_slots
    global parking_history

    vehicle_number = vehicle_number.strip().upper()

    vehicle = parking_slots[
        parking_slots["Vehicle_Number"] == vehicle_number
    ]

    if len(vehicle) == 0:
        return (
            f"Vehicle {vehicle_number} was not found.",
            parking_slots,
            parking_history
        )

    index = vehicle.index[0]

    slot_id = parking_slots.loc[index, "Slot_ID"]

    entry_time = datetime.strptime(
        parking_slots.loc[index, "Entry_Time"],
        "%Y-%m-%d %H:%M:%S"
    )

    exit_time = datetime.now()

    parking_hours = (
        exit_time - entry_time
    ).total_seconds() / 3600

    parking_fee = calculate_fee(parking_hours)

    new_record = pd.DataFrame([{
        "Vehicle_Number": vehicle_number,
        "Slot_ID": slot_id,
        "Entry_Time": entry_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
        "Exit_Time": exit_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
        "Parking_Hours": round(parking_hours, 2),
        "Parking_Fee": parking_fee
    }])

    parking_history = pd.concat(
        [parking_history, new_record],
        ignore_index=True
    )

    parking_slots.loc[index, "Status"] = "Available"
    parking_slots.loc[index, "Vehicle_Number"] = "-"
    parking_slots.loc[index, "Entry_Time"] = "-"
    parking_slots.loc[index, "Exit_Time"] = exit_time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
    parking_slots.loc[index, "Parking_Hours"] = round(
        parking_hours, 2
    )
    parking_slots.loc[index, "Parking_Fee"] = parking_fee

    message = (
        f"Vehicle Exit Successful!\n\n"
        f"Vehicle Number: {vehicle_number}\n"
        f"Parking Slot: {slot_id}\n"
        f"Parking Duration: {parking_hours:.2f} hours\n"
        f"Parking Fee: ₹{parking_fee}"
    )

    return (
        message,
        parking_slots,
        parking_history
    )

In [23]:
def find_slot():

    available = get_available_slots()

    if len(available) == 0:
        message = "No parking slots are currently available."
    else:
        message = (
            "Available Parking Slots:\n\n"
            + ", ".join(available)
        )

    return message

In [24]:
def live_dashboard():

    total, available, occupied, occupancy, revenue = get_dashboard()

    summary = f"""
# Smart Parking Dashboard

### Parking Statistics

**Total Slots:** {total}

**Available Slots:** {available}

**Occupied Slots:** {occupied}

**Occupancy Rate:** {occupancy:.2f}%

**Total Revenue:** ₹{revenue:.2f}
"""

    status_table = parking_slots[
        [
            "Slot_ID",
            "Status",
            "Vehicle_Number",
            "Entry_Time",
            "Parking_Hours",
            "Parking_Fee"
        ]
    ].copy()

    fig = plt.figure(figsize=(7, 5))

    plt.bar(
        ["Available", "Occupied"],
        [available, occupied]
    )

    plt.title("Parking Slot Occupancy")
    plt.xlabel("Status")
    plt.ylabel("Number of Slots")

    plt.tight_layout()

    return summary, status_table, fig

In [25]:
def show_history():

    if len(parking_history) == 0:

        empty_message = "No parking history available yet."

        return empty_message, parking_history

    return "Parking History", parking_history

In [26]:
def simulate_sensors():

    global parking_slots

    occupied_count = 0

    for index in parking_slots.index:

        if parking_slots.loc[index, "Status"] == "Occupied":
            occupied_count += 1

    sensor_records = []

    for _, row in parking_slots.iterrows():

        if row["Status"] == "Occupied":
            sensor_status = "Vehicle Detected"
        else:
            sensor_status = "No Vehicle"

        sensor_records.append({
            "Slot_ID": row["Slot_ID"],
            "Sensor_Status": sensor_status,
            "Timestamp": datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        })

    sensor_df = pd.DataFrame(sensor_records)

    sensor_df.to_csv(
        "parking_sensor_data.csv",
        index=False
    )

    return (
        "IoT sensor data successfully updated.",
        sensor_df
    )

In [27]:
def generate_report():

    report_file = "Smart_Parking_Report.xlsx"

    with pd.ExcelWriter(
        report_file,
        engine="openpyxl"
    ) as writer:

        parking_slots.to_excel(
            writer,
            sheet_name="Current Parking",
            index=False
        )

        parking_history.to_excel(
            writer,
            sheet_name="Parking History",
            index=False
        )

    return report_file

In [28]:
with gr.Blocks(
    title="Cloud-Based Smart Parking System"
) as app:

    gr.Markdown(
        """
        # CLOUD-BASED SMART PARKING SYSTEM

        A smart parking management application using
        simulated IoT sensors, centralized cloud-style
        data management and real-time analytics.
        """
    )

    with gr.Tab("Vehicle Entry"):

        gr.Markdown(
            "### Register a vehicle and automatically allocate a parking slot."
        )

        entry_vehicle = gr.Textbox(
            label="Vehicle Number",
            placeholder="Example: TN01AB1234"
        )

        entry_button = gr.Button(
            "Vehicle Entry"
        )

        entry_message = gr.Textbox(
            label="Entry Status",
            lines=5
        )

        entry_table = gr.Dataframe(
            value=parking_slots,
            label="Current Parking Status"
        )

        entry_button.click(
            vehicle_entry,
            inputs=entry_vehicle,
            outputs=[
                entry_message,
                entry_table
            ]
        )


    with gr.Tab("Vehicle Exit"):

        gr.Markdown(
            "### Process vehicle exit and calculate the parking fee."
        )

        exit_vehicle = gr.Textbox(
            label="Vehicle Number",
            placeholder="Example: TN01AB1234"
        )

        exit_button = gr.Button(
            "Vehicle Exit"
        )

        exit_message = gr.Textbox(
            label="Exit Status",
            lines=6
        )

        exit_table = gr.Dataframe(
            value=parking_slots,
            label="Current Parking Status"
        )

        history_table = gr.Dataframe(
            value=parking_history,
            label="Parking History"
        )

        exit_button.click(
            vehicle_exit,
            inputs=exit_vehicle,
            outputs=[
                exit_message,
                exit_table,
                history_table
            ]
        )


    with gr.Tab("Find Slot"):

        gr.Markdown(
            "### Find currently available parking slots."
        )

        find_button = gr.Button(
            "Find Available Slot"
        )

        find_result = gr.Textbox(
            label="Available Slots",
            lines=5
        )

        find_button.click(
            find_slot,
            outputs=find_result
        )


    with gr.Tab("Live Dashboard"):

        dashboard_button = gr.Button(
            "Refresh Live Dashboard"
        )

        dashboard_summary = gr.Markdown()

        dashboard_table = gr.Dataframe(
            label="Live Parking Status"
        )

        dashboard_chart = gr.Plot(
            label="Parking Occupancy"
        )

        dashboard_button.click(
            live_dashboard,
            outputs=[
                dashboard_summary,
                dashboard_table,
                dashboard_chart
            ]
        )


    with gr.Tab("Parking History"):

        history_button = gr.Button(
            "View Parking History"
        )

        history_message = gr.Textbox(
            label="Status"
        )

        history_output = gr.Dataframe(
            label="Parking History"
        )

        history_button.click(
            show_history,
            outputs=[
                history_message,
                history_output
            ]
        )


    with gr.Tab("IoT Sensors"):

        gr.Markdown(
            """
            ### Simulated Smart Parking Sensors

            Each parking slot represents an IoT sensor that
            detects whether a vehicle is present.
            """
        )

        sensor_button = gr.Button(
            "Simulate Sensors"
        )

        sensor_message = gr.Textbox(
            label="Sensor Status"
        )

        sensor_output = gr.Dataframe(
            label="Sensor Data"
        )

        sensor_button.click(
            simulate_sensors,
            outputs=[
                sensor_message,
                sensor_output
            ]
        )


    with gr.Tab("Reports"):

        gr.Markdown(
            """
            ### Parking Report

            Generate an Excel report containing current
            parking status and complete parking history.
            """
        )

        report_button = gr.Button(
            "Generate & Download Report"
        )

        report_file = gr.File(
            label="Download Report"
        )

        report_button.click(
            generate_report,
            outputs=report_file
        )

print("Interactive application created successfully!")

Interactive application created successfully!


In [29]:
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1109487a9f57d604dc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
